# 03 — Hybrid BERT–LDA

Concatenate LDA document-topic vectors `θ` with Sentence-BERT embeddings `e`:

`h = [ α · θ  |  (1−α) · e ]`

Then UMAP + k-means + c-TF-IDF (George & Sumathy; Paul et al.). `α` is the abstract's tunable blend weight.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import DEFAULT_N_DOCS, GAMMA, RANDOM_SEED
from src.data import load_corpus
from src.preprocess import preprocess_frame
from src.lda_baseline import fit_lda
from src.embeddings import encode_documents
from src.hybrid import build_hybrid_matrix, fit_cluster_topics
from src.evaluate import topic_coherence, clustering_silhouette, summarize_topics

In [ ]:
df = preprocess_frame(load_corpus("arxiv", n_docs=DEFAULT_N_DOCS, seed=RANDOM_SEED))
texts = df["tokens"].tolist()
k = 12
lda = fit_lda(texts, n_topics=k, seed=RANDOM_SEED)
emb = encode_documents(df["text"].tolist(), corpus="arxiv", seed=RANDOM_SEED)
hybrid = build_hybrid_matrix(lda.theta, emb, gamma=GAMMA)
print("theta", lda.theta.shape, "emb", emb.shape, "hybrid", hybrid.shape)

hyb = fit_cluster_topics(hybrid, texts, n_topics=k, reducer="umap")
print("Hybrid C_v", topic_coherence(hyb.topic_words, texts))
print("Silhouette", clustering_silhouette(hyb.reduced, hyb.labels))
summarize_topics(hyb.topic_words)

Compare `α=0` (pure embeddings), `α=0.5` (hybrid), and `α=1` (pure LDA geometry) in notebook 04 / `run_pipeline.py`. The α that maximises `C_v` is the coherence-driven weighting from the project abstract.